In [1]:
from IPython.display import clear_output

# install nnsight to analyze neural network internals
!pip install -U nnsight

clear_output()

In [2]:
from huggingface_hub import notebook_login

notebook_login()

/Users/ana.serpa/cs-oolong-research/.venv2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# load model
import nnsight

model = nnsight.LanguageModel(
    'meta-llama/Llama-3.2-1B',
    device_map='auto'
)

In [5]:
# does our model know where Paris is?
import torch

base_prompt = "Paris is in the country of"

with torch.no_grad():
    with model.trace(base_prompt):
        # get logits over final token
        base_logits = model.output.logits[:, -1, :].save()

print('Prompt:', base_prompt)
print(f'Model completion:{model.tokenizer.decode(base_logits.argmax(dim=-1))}')

Loading weights: 100%|███████████| 146/146 [00:00<00:00, 38037.67it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Prompt: Paris is in the country of
Model completion: France


In [6]:
base_prompt = "Paris is in the country of"
source_prompt = "Rio is in the country of"

In [9]:
# 1. collect source activations
source_hidden_states = []

with torch.no_grad():
    with model.trace(source_prompt):
        # get hidden states of all layers in the network
        # we index the output at 0 because it's a tuple where the first index is the hidden state
        for layer in model.model.layers:
            source_hidden_states.append(layer.output[0].save())

In [10]:
# 2 and 3. intervene and collect intervention results

# we'll keep track of the probability of outputing Brazil vs. the probability of outputing France
source_country = model.tokenizer(" Brazil")["input_ids"][1] # includes a space
base_country = model.tokenizer(" France")["input_ids"][1] # includes a space

num_tokens = len(model.tokenizer(base_prompt).input_ids) # get number of tokens in prompt

causal_effects = []
# iterate through all the layers
for layer_idx in range(model.config.num_hidden_layers):
    causal_effect_per_layer = []
    # iterate through all tokens
    for token_idx in range(num_tokens):
        with torch.no_grad():
            with model.trace(base_prompt) as tracer:
                # 2. change the value of the base activation to the source value
                model.model.layers[layer_idx].output[0][:, token_idx, :] = \
                    source_hidden_states[layer_idx][:, token_idx, :]

                # 3. get intervened output & compare to base output
                intervened_logits = model.output.logits[:, -1, :]
                intervened_probs = intervened_logits.softmax(dim=-1)
                # in this case, we'll keep track of how much we changed
                # towards Brazil & away from France -> the higher, the more causal effect!
                intervened_prob_diff = (intervened_probs[0, source_country] - intervened_probs[0, base_country]).item().save()

            causal_effect_per_layer.append(intervened_prob_diff)
    causal_effects.append(causal_effect_per_layer)

Traceback (most recent call last):
  File "/Users/ana.serpa/cs-oolong-research/.venv2/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3748, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/gd/nzmy_7gd6pdg32k7kvyhkt1m0000gq/T/ipykernel_1992/3653059987.py", line 16, in <module>
    with model.trace(base_prompt) as tracer:
  File "/var/folders/gd/nzmy_7gd6pdg32k7kvyhkt1m0000gq/T/ipykernel_1992/3653059987.py", line 19, in <module>
    source_hidden_states[layer_idx][:, token_idx, :]

IndexError: too many indices for tensor of dimension 2
